# MRVA Drought And Recovery Metrics

This notebook computes five groundwater drought and recovery metrics from the final reconstructed WTD product. WTD is measured in meters below land surface, so larger WTD means a deeper water table.

## Main Metrics
| # | Variable | Figure label | Meaning |
|---|---|---|---|
| 1 | `decline_m` | 2012 drought WTD decline | Maximum short-term water-table deepening during the 2012 drought |
| 2 | `Rdown_m_per_month` | Drought deepening rate | Rate of WTD deepening from the pre-drought shallow-water state to the drought peak |
| 3 | `RR_early` | Post-drought recovery fraction by spring 2013 | Fraction of the 2012 drought decline recovered from November 2012 through April 2013 |
| 4 | `T50_months` | Time to 50% recovery | Number of months after the drought peak required to recover half of the 2012 decline |
| 5 | `RR2019` | Long-term recovery fraction by 2019 | Fraction of the 2012 long-term deficit recovered by the 2019 mean WTD after several wet years |

## Notes
- Recovery fractions are saved as raw, unclipped values, so values smaller than 0 or larger than 1 preserve physical interpretation.
- Maps control the displayed range with color limits (`vmin`/`vmax`) rather than clipping the metric data.
- The notebook saves a stable CSV table for later Fig2/Fig3/Fig4 workflows.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False

ROOT = next(
    p for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    if (p / 'outputs' / 'RECON_MAIN_2011_2023').exists()
)
RECON = ROOT / 'outputs' / 'RECON_MAIN_2011_2023'
OUT_DIR = RECON / 'metrics'
OUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_CSV = OUT_DIR / 'drought_metrics.csv'

def display_path(path):
    try:
        return str(Path(path).resolve().relative_to(ROOT))
    except ValueError:
        return str(path)

mat = np.load(RECON / 'reconstruction' / 'wtd_reconstructed_matrix.npy', mmap_mode='r')
MI = pd.read_csv(RECON / 'metadata' / 'month_index.csv')
MI['month_label'] = MI['month_label'].astype(str)
MI['year'] = MI['month_label'].str[:4].astype(int)
MI['month'] = MI['month_label'].str[5:7].astype(int)
GL = pd.read_csv(RECON / 'metadata' / 'grid_lookup.csv')

n_month, n_grid = mat.shape
if len(GL) != n_grid:
    raise ValueError(f'grid_lookup rows ({len(GL)}) != matrix columns ({n_grid})')

print(f'Reconstruction matrix: {n_month} months x {n_grid} grid cells')
print(f'Output table: {display_path(METRICS_CSV)}')


## 1. Time Windows And Parameters


In [ ]:
# Key parameters. Change here if the paper definition changes.
PRE_DROUGHT_MONTHS = (1, 4)      # 2012 Jan-Apr: shallow pre-drought reference
DROUGHT_MONTHS = (5, 10)         # 2012 May-Oct: growing-season drought response
RECOVERY_START = '2012-11'
RECOVERY_END = '2013-04'
LONGTERM_BASELINE_YEAR = 2011
LONGTERM_DEFICIT_YEAR = 2012
LONGTERM_RECOVERY_YEAR = 2019
MIN_DECLINE_M = 0.30
MIN_LONGTERM_DEFICIT_M = 0.10

def month_idx_for_year_months(year, start_month, end_month):
    mask = (MI.year == year) & MI.month.between(start_month, end_month)
    idx = MI.loc[mask, 'month_idx'].to_numpy(dtype=int)
    if idx.size == 0:
        raise ValueError(f'No months found for {year}-{start_month:02d}..{end_month:02d}')
    return idx

def month_idx_for_year(year):
    idx = MI.loc[MI.year == year, 'month_idx'].to_numpy(dtype=int)
    if idx.size == 0:
        raise ValueError(f'No months found for {year}')
    return idx

def month_idx_for_label_range(start_label, end_label):
    mask = MI.month_label.between(start_label, end_label)
    idx = MI.loc[mask, 'month_idx'].to_numpy(dtype=int)
    if idx.size == 0:
        raise ValueError(f'No months found for {start_label}..{end_label}')
    return idx

def idx_labels(idx):
    return ', '.join(MI.set_index('month_idx').loc[idx, 'month_label'].astype(str).tolist())

pre = month_idx_for_year_months(2012, *PRE_DROUGHT_MONTHS)
drought = month_idx_for_year_months(2012, *DROUGHT_MONTHS)
recovery = month_idx_for_label_range(RECOVERY_START, RECOVERY_END)
base_year = month_idx_for_year(LONGTERM_BASELINE_YEAR)
long_deficit_year = month_idx_for_year(LONGTERM_DEFICIT_YEAR)
long_recovery_year = month_idx_for_year(LONGTERM_RECOVERY_YEAR)



## 2. Compute Five Metrics


In [ ]:
# 1. Short-term drought decline: deepest WTD during drought minus shallowest pre-drought WTD.
pre_wtd = np.asarray(mat[pre], dtype=np.float32)
drought_wtd = np.asarray(mat[drought], dtype=np.float32)
recovery_window_wtd = np.asarray(mat[recovery], dtype=np.float32)

pre_drought_wtd = np.nanmin(pre_wtd, axis=0)
pre_drought_month_idx = pre[np.nanargmin(pre_wtd, axis=0)]

drought_peak_wtd = np.nanmax(drought_wtd, axis=0)
drought_peak_month_idx = drought[np.nanargmax(drought_wtd, axis=0)]

decline = drought_peak_wtd - pre_drought_wtd
valid_decline = decline >= MIN_DECLINE_M

# 2. Short-term deepening rate.
months_to_peak = np.maximum(drought_peak_month_idx - pre_drought_month_idx, 1)
decline_rate = decline / months_to_peak

# 3. Post-drought recovery fraction before the 2013 growing-season drawdown.
recovery_wtd = np.nanmin(recovery_window_wtd, axis=0)
recovery_month_idx = recovery[np.nanargmin(recovery_window_wtd, axis=0)]
recovery_amount = drought_peak_wtd - recovery_wtd
with np.errstate(divide='ignore', invalid='ignore'):
    early_recovery_fraction = recovery_amount / decline

# 4. Time to 50% recovery after the 2012 drought peak.
t50_threshold_wtd = drought_peak_wtd - 0.5 * decline
month_index_grid = np.arange(n_month, dtype=int)[:, None]
all_wtd = np.asarray(mat, dtype=np.float32)
t50_hit = (month_index_grid > drought_peak_month_idx[None, :]) & (all_wtd <= t50_threshold_wtd[None, :])
t50_first_idx = t50_hit.argmax(axis=0)
t50_months = (t50_first_idx - drought_peak_month_idx).astype(float)
t50_months[~t50_hit.any(axis=0)] = np.nan

# 5. Long-term recovery by 2019 after several wet years.
baseline_wtd = np.nanmean(np.asarray(mat[base_year], dtype=np.float32), axis=0)
longterm_peak_wtd = np.nanmax(np.asarray(mat[long_deficit_year], dtype=np.float32), axis=0)
longterm_deficit = longterm_peak_wtd - baseline_wtd
residual_deficit_2019 = np.nanmean(np.asarray(mat[long_recovery_year], dtype=np.float32), axis=0) - baseline_wtd
valid_longterm_deficit = longterm_deficit >= MIN_LONGTERM_DEFICIT_M
with np.errstate(divide='ignore', invalid='ignore'):
    rr2019 = (longterm_deficit - residual_deficit_2019) / longterm_deficit

# Mask unstable short-term quantities where the 2012 drought response is too small.
short_term_arrays = [
    decline,
    decline_rate,
    recovery_amount,
    early_recovery_fraction,
    t50_months,
]
for arr in short_term_arrays:
    arr[~valid_decline] = np.nan

# Mask unstable long-term recovery fractions where long-term deficit is too small.
for arr in [rr2019]:
    arr[~valid_longterm_deficit] = np.nan

print('Computed five drought/recovery metrics.')


## 3. Save Metric Table


In [ ]:
metrics_df = GL.copy()
if 'grid_id' not in metrics_df.columns:
    metrics_df.insert(0, 'grid_id', np.arange(n_grid, dtype=int))

metrics_df['pre_wtd_m'] = pre_drought_wtd
metrics_df['drought_peak_wtd_m'] = drought_peak_wtd
metrics_df['pre_month_idx'] = pre_drought_month_idx
metrics_df['drought_peak_month_idx'] = drought_peak_month_idx
metrics_df['months_to_peak'] = months_to_peak
metrics_df['decline_m'] = decline
metrics_df['Rdown_m_per_month'] = decline_rate
metrics_df['early_recovery_wtd_m'] = recovery_wtd
metrics_df['early_recovery_month_idx'] = recovery_month_idx
metrics_df['early_recovery_m'] = recovery_amount
metrics_df['RR_early'] = early_recovery_fraction
metrics_df['T50_months'] = t50_months
metrics_df['baseline_wtd_2011_m'] = baseline_wtd
metrics_df['annual_peak_wtd_2012_m'] = longterm_peak_wtd
metrics_df['deficit_2012_m'] = longterm_deficit
metrics_df['residual_deficit_2019_m'] = residual_deficit_2019
metrics_df['RR2019'] = rr2019
metrics_df['valid_decline'] = valid_decline
metrics_df['valid_deficit_2012'] = valid_longterm_deficit

metrics_df.to_csv(METRICS_CSV, index=False)
print('Saved:', display_path(METRICS_CSV))
metrics_df.shape


## 4. Spatial Patterns


In [ ]:
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'],
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
    'font.size': 7,
    'axes.linewidth': 0.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
    'xtick.major.size': 2.0,
    'ytick.major.size': 2.0,
    'legend.frameon': False,
})

LOSS_CMAP = LinearSegmentedColormap.from_list('mrva_loss', ['#F7F7F7', '#F2B07D', '#B64342'])
RECOVERY_CMAP = LinearSegmentedColormap.from_list('mrva_recovery', ['#F7F7F7', '#A6D6D4', '#0F4D92'])
MEMORY_CMAP = LinearSegmentedColormap.from_list('mrva_memory', ['#F7F7F7', '#C9B8D8', '#6F4A8E'])
for cmap in [LOSS_CMAP, RECOVERY_CMAP, MEMORY_CMAP]:
    cmap.set_bad('#F0F0F0')

PANEL_LABEL_KW = dict(fontsize=8, fontweight='bold', ha='left', va='top')

metric_plot_specs = [
    {
        'key': 'decline_m',
        'short': 'Drought decline',
        'window': 'May-Oct 2012',
        'cbar': 'm',
        'cmap': LOSS_CMAP,
        'vmin': 0.0,
        'vmax_pct': 98.5,
    },
    {
        'key': 'Rdown_m_per_month',
        'short': 'Deepening rate',
        'window': 'pre-drought to peak',
        'cbar': 'm month$^{-1}$',
        'cmap': LOSS_CMAP,
        'vmin': 0.0,
        'vmax_pct': 98.5,
    },
    {
        'key': 'RR_early',
        'short': 'Post-drought recovery',
        'window': 'Nov 2012-Apr 2013',
        'cbar': 'fraction',
        'cmap': RECOVERY_CMAP,
        'vmin': 0.0,
        'vmax': 1.0,
    },
    {
        'key': 'T50_months',
        'short': 'Half-recovery time',
        'window': 'after 2012 peak',
        'cbar': 'months',
        'cmap': MEMORY_CMAP,
        'vmin': 0.0,
        'vmax_pct': 98.0,
    },
    {
        'key': 'RR2019',
        'short': 'Long-term recovery',
        'window': '2019 mean state',
        'cbar': 'fraction',
        'cmap': RECOVERY_CMAP,
        'vmin': 0.0,
        'vmax': 1.0,
    },
]
rows_arr = metrics_df['row'].to_numpy(dtype=int)
cols_arr = metrics_df['col'].to_numpy(dtype=int)
n_rows = int(rows_arr.max()) + 1
n_cols = int(cols_arr.max()) + 1
origin_mode = 'upper' if metrics_df[['row', 'y']].corr().loc['row', 'y'] < 0 else 'lower'
mask_grid = np.zeros((n_rows, n_cols), dtype=float)
mask_grid[rows_arr, cols_arr] = 1.0

def rasterize_metric(values):
    arr = np.full((n_rows, n_cols), np.nan, dtype=np.float32)
    arr[rows_arr, cols_arr] = np.asarray(values, dtype=np.float32)
    return arr

def finite_limits(values, spec):
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    vmin = spec.get('vmin')
    vmax = spec.get('vmax')
    if vmax is None:
        vmax = float(np.nanpercentile(finite, spec.get('vmax_pct', 98))) if finite.size else 1.0
    if vmin is None:
        vmin = float(np.nanpercentile(finite, spec.get('vmin_pct', 2))) if finite.size else 0.0
    if not np.isfinite(vmax) or vmax <= vmin:
        vmax = vmin + 1.0
    return vmin, vmax

def colorbar_extend(values, vmin, vmax):
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if not finite.size:
        return 'neither'
    has_low = np.nanmin(finite) < vmin
    has_high = np.nanmax(finite) > vmax
    if has_low and has_high:
        return 'both'
    if has_low:
        return 'min'
    if has_high:
        return 'max'
    return 'neither'

def add_panel_label(ax, label):
    ax.text(-0.02, 1.03, label, transform=ax.transAxes, **PANEL_LABEL_KW)

def add_metric_colorbar(fig, ax, im, label, extend='neither'):
    cax = inset_axes(ax, width='58%', height='3.8%', loc='lower left', borderpad=0.72)
    cbar = fig.colorbar(im, cax=cax, orientation='horizontal', extend=extend)
    cbar.outline.set_linewidth(0.4)
    cbar.ax.tick_params(labelsize=5.6, length=1.6, width=0.4, pad=0.5)
    cbar.ax.set_title(label, fontsize=5.6, pad=1.2)
    return cbar

def export_figure(fig, stem, dpi=600):
    out = (FIG_DIR / stem).with_suffix('.png')
    fig.savefig(out, bbox_inches='tight', dpi=dpi)
    return [out]

fig, axes = plt.subplots(1, 5, figsize=(11.2, 3.0), constrained_layout=False)
fig.subplots_adjust(left=0.015, right=0.995, bottom=0.07, top=0.88, wspace=0.035)

for panel_i, (ax, spec) in enumerate(zip(axes, metric_plot_specs)):
    values = metrics_df[spec['key']].to_numpy(dtype=float)
    arr = rasterize_metric(values)
    vmin, vmax = finite_limits(values, spec)
    extend = colorbar_extend(values, vmin, vmax)
    im = ax.imshow(arr, origin=origin_mode, cmap=spec['cmap'], vmin=vmin, vmax=vmax, interpolation='nearest')
    ax.contour(mask_grid, levels=[0.5], colors='#333333', linewidths=0.25, origin=origin_mode)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect('equal')
    ax.set_title(f"{spec['short']}\n{spec['window']}", fontsize=7.0, pad=2.2)
    add_panel_label(ax, chr(ord('a') + panel_i))
    add_metric_colorbar(fig, ax, im, spec['cbar'], extend=extend)
    for spine in ax.spines.values():
        spine.set_visible(False)
saved = export_figure(fig, 'drought_metric_spatial')
print('Saved figure exports:')
for out in saved:
    print(' ', display_path(out))
plt.show()